# SpaceX-like Booster Landing Simulation

This standalone notebook simulates a simplified three-dimensional reusable booster landing. It is an educational model, not a reconstruction of Falcon 9 flight software or vehicle data.

The controller follows a stopping-distance vertical-speed envelope while steering toward a landing pad. The simulation enforces maximum thrust, minimum running throttle, gimbal angle, dry mass, and ground-contact constraints.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

plt.rcParams.update({"figure.figsize": (11, 6.5), "axes.grid": True})

@dataclass(frozen=True)
class BoosterConfig:
    gravity: float = 9.80665          # m/s^2
    initial_mass: float = 32_000.0   # kg
    dry_mass: float = 18_000.0       # kg
    max_thrust: float = 760_000.0    # N, one-engine equivalent
    min_throttle: float = 0.20       # fraction of maximum thrust
    specific_impulse: float = 282.0  # s
    max_gimbal_deg: float = 18.0     # deg from vertical
    cone_half_angle_deg: float = 30.0  # 3-D glide-slope cone
    landing_pad_radius: float = 1.0     # m, cone tip tolerance
    dt: float = 0.02                 # s
    max_time: float = 120.0          # s

cfg = BoosterConfig()

# State = [x, y, z, vx, vy, vz, mass]
initial_state = np.array([700.0, -500.0, 1800.0, -25.0, 15.0, -85.0, cfg.initial_mass])
landing_pad = np.array([0.0, 0.0])

## Guidance and constrained thrust command

The vertical target speed is derived from $v^2 = 2 a h$. This makes the commanded descent speed shrink continuously near the ground. Lateral guidance commands velocity toward the pad. The resulting acceleration is converted to a thrust vector and projected onto the engine limits.

In [ ]:
def guidance_command(state, config):
    x, y, z, vx, vy, vz, mass = state

    # Stopping-speed envelope: fast high up, progressively slower near the pad.
    soft_deceleration = 1.0  # m/s^2; shapes the approach rather than engine capability
    vertical_speed_target = -max(
        1.5,
        min(70.0, np.sqrt(max(2.0 * soft_deceleration * z, 0.0))),
    )
    horizontal_error = np.array([x, y]) - landing_pad
    horizontal_speed_target = np.clip(-0.08 * horizontal_error, -30.0, 30.0)

    # Velocity-level CBF for the glide-slope cone:
    # h = tan(gamma) * z - ||horizontal error|| >= 0.
    horizontal_distance = np.linalg.norm(horizontal_error)
    cone_tangent = np.tan(np.deg2rad(config.cone_half_angle_deg))
    cone_margin = config.landing_pad_radius + cone_tangent * z - horizontal_distance
    if horizontal_distance > 1e-9:
        radial_direction = horizontal_error / horizontal_distance
        target_radial_speed = radial_direction @ horizontal_speed_target
        max_radial_speed = cone_tangent * vertical_speed_target + 0.25 * cone_margin
        if target_radial_speed > max_radial_speed:
            horizontal_speed_target += (max_radial_speed - target_radial_speed) * radial_direction

    desired_axy = 0.90 * (horizontal_speed_target - np.array([vx, vy]))
    desired_az = 1.40 * (vertical_speed_target - vz)

    # Thrust acceleration includes gravity compensation. Negative vertical thrust
    # is impossible, so the engine may coast when no upward force is requested.
    thrust_accel = np.array([*desired_axy, max(0.0, desired_az + config.gravity)])
    requested_accel = np.linalg.norm(thrust_accel)

    if requested_accel < 1e-9:
        return np.zeros(3), 0.0, 0.0, vertical_speed_target, horizontal_speed_target

    max_gimbal = np.deg2rad(config.max_gimbal_deg)
    horizontal_accel = thrust_accel[:2]
    horizontal_norm = np.linalg.norm(horizontal_accel)
    gimbal = min(np.arctan2(horizontal_norm, thrust_accel[2]), max_gimbal)
    horizontal_direction = (
        horizontal_accel / horizontal_norm if horizontal_norm > 1e-9 else np.zeros(2)
    )
    requested_thrust = mass * requested_accel
    min_thrust = config.min_throttle * config.max_thrust
    thrust_magnitude = np.clip(requested_thrust, min_thrust, config.max_thrust)
    thrust_direction = np.array([
        *(np.sin(gimbal) * horizontal_direction), np.cos(gimbal),
    ])
    thrust = thrust_magnitude * thrust_direction
    throttle = thrust_magnitude / config.max_thrust
    return thrust, throttle, gimbal, vertical_speed_target, horizontal_speed_target

In [ ]:
def simulate_landing(state0, config):
    state = np.asarray(state0, dtype=float).copy()
    history = []
    steps = int(config.max_time / config.dt)

    for step in range(steps + 1):
        time = step * config.dt
        x, y, z, vx, vy, vz, mass = state
        thrust, throttle, gimbal, vz_target, vxy_target = guidance_command(state, config)
        history.append([time, *state, *thrust, throttle, gimbal, vz_target, *vxy_target])

        if z <= 0.0 or mass <= config.dry_mass:
            break

        acceleration = thrust / mass + np.array([0.0, 0.0, -config.gravity])
        mass_rate = -np.linalg.norm(thrust) / (config.specific_impulse * config.gravity)

        # Constant-acceleration position update and forward velocity/mass update.
        next_state = state.copy()
        next_state[0:3] += config.dt * state[3:6] + 0.5 * config.dt**2 * acceleration
        next_state[3:6] += config.dt * acceleration
        next_state[6] = max(config.dry_mass, mass + config.dt * mass_rate)
        state = next_state

    data = np.asarray(history)
    columns = {name: i for i, name in enumerate([
        "t", "x", "y", "z", "vx", "vy", "vz", "mass",
        "tx", "ty", "tz", "throttle", "gimbal",
        "vz_target", "vx_target", "vy_target",
    ])}
    return data, columns

result, col = simulate_landing(initial_state, cfg)

In [ ]:
final = result[-1]
touchdown_speed = np.linalg.norm(final[[col["vx"], col["vy"], col["vz"]]])
fuel_used = cfg.initial_mass - final[col["mass"]]
max_gimbal_used = np.max(np.abs(np.rad2deg(result[:, col["gimbal"]])))
max_throttle_used = np.max(result[:, col["throttle"]])
landed = final[col["z"]] <= 0.5
horizontal_distance = np.hypot(result[:, col["x"]], result[:, col["y"]])
cone_margin = (
    cfg.landing_pad_radius
    + np.tan(np.deg2rad(cfg.cone_half_angle_deg)) * np.maximum(result[:, col["z"]], 0.0)
    - horizontal_distance
)

print(f"Status: {'LANDED' if landed else 'STOPPED BEFORE LANDING'}")
print(f"Flight time:          {final[col['t']]:8.2f} s")
print(f"Final pad error:      x={final[col['x']]:7.2f}, y={final[col['y']]:7.2f} m")
print(f"Final velocity:       vx={final[col['vx']]:7.2f}, vy={final[col['vy']]:7.2f}, vz={final[col['vz']]:7.2f} m/s")
print(f"Touchdown speed:      {touchdown_speed:8.2f} m/s")
print(f"Propellant consumed:  {fuel_used:8.1f} kg")
print(f"Maximum throttle:     {100 * max_throttle_used:8.1f} %")
print(f"Maximum gimbal:       {max_gimbal_used:8.2f} deg")
print(f"Minimum cone margin:  {np.min(cone_margin):8.2f} m")

assert max_gimbal_used <= cfg.max_gimbal_deg + 1e-9
assert np.min(result[:, col["mass"]]) >= cfg.dry_mass - 1e-9
assert max_throttle_used <= 1.0 + 1e-9
assert np.min(cone_margin) >= -0.05  # numerical integration tolerance

In [ ]:
fig = plt.figure(figsize=(15, 11), constrained_layout=True)
ax_traj = fig.add_subplot(2, 2, 1, projection="3d")
ax_velocity = fig.add_subplot(2, 2, 2)
ax_engine = fig.add_subplot(2, 2, 3)
ax_mass = fig.add_subplot(2, 2, 4)

# Landing trajectory with sampled thrust arrows.
ax = ax_traj
ax.plot(result[:, col["x"]], result[:, col["y"]], result[:, col["z"]], linewidth=2.2, label="Booster trajectory")
ground_extent = 380.0
ground_axis = np.linspace(-ground_extent, ground_extent, 8)
ground_x, ground_y = np.meshgrid(ground_axis, ground_axis)
ax.plot_wireframe(ground_x, ground_y, np.zeros_like(ground_x), linewidth=0.5, alpha=0.35)
cone_z = np.linspace(0.0, initial_state[2], 24)
cone_theta = np.linspace(0.0, 2.0 * np.pi, 28)
cone_z_grid, cone_theta_grid = np.meshgrid(cone_z, cone_theta)
cone_radius = cfg.landing_pad_radius + np.tan(np.deg2rad(cfg.cone_half_angle_deg)) * cone_z_grid
ax.plot_wireframe(cone_radius * np.cos(cone_theta_grid),
                  cone_radius * np.sin(cone_theta_grid), cone_z_grid,
                  rstride=5, cstride=5, linewidth=0.45, alpha=0.25)
ax.scatter([landing_pad[0]], [landing_pad[1]], [0.0], marker="*", s=160, label="Landing pad")
arrow_indices = np.linspace(0, len(result) - 1, 16).round().astype(int)
for i in arrow_indices:
    thrust = result[i, [col["tx"], col["ty"], col["tz"]]]
    norm = np.linalg.norm(thrust)
    if norm > 1e-9:
        direction = thrust / norm
        ax.quiver(result[i, col["x"]], result[i, col["y"]], result[i, col["z"]],
                  direction[0], direction[1], direction[2],
                  length=70.0, normalize=True, arrow_length_ratio=0.20, linewidth=1.2)
ax.plot([], [], linewidth=1.5, label="Thrust direction / cone constraint")
ax.set_xlabel("X [m]")
ax.set_ylabel("Y [m]")
ax.set_zlabel("Altitude [m]")
ax.set_title("3-D booster landing")
ax.view_init(elev=24, azim=-55)
ax.legend()

ax = ax_velocity
ax.plot(result[:, col["t"]], result[:, col["vz"]], label="Vertical speed")
ax.plot(result[:, col["t"]], result[:, col["vz_target"]], linestyle="--", label="Stopping-speed target")
ax.plot(result[:, col["t"]], result[:, col["vx"]], label="Horizontal speed")
ax.plot(result[:, col["t"]], result[:, col["vy"]], label="Crossrange speed")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Velocity [m/s]")
ax.set_title("Velocity guidance")
ax.legend()

ax = ax_engine
ax.plot(result[:, col["t"]], 100.0 * result[:, col["throttle"]], label="Throttle")
ax.plot(result[:, col["t"]], np.rad2deg(result[:, col["gimbal"]]), label="Gimbal angle")
ax.axhline(100.0 * cfg.min_throttle, linestyle="--", label="Minimum running throttle")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Throttle [%] / gimbal [deg]")
ax.set_title("Engine command")
ax.legend()

ax = ax_mass
ax.plot(result[:, col["t"]], result[:, col["mass"]], label="Vehicle mass")
ax.axhline(cfg.dry_mass, linestyle="--", label="Dry mass")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Mass [kg]")
ax.set_title("Propellant consumption")
ax.legend()

fig.suptitle("SpaceX-like reusable booster landing simulation", fontsize=15)
plt.show()

## Experiments

Try changing `initial_state`, `max_gimbal_deg`, `max_thrust`, or the `soft_deceleration` value. More demanding initial conditions can make the landing infeasible because the simplified controller cannot violate thrust, fuel, or gimbal limits.